# 💾 Stage 3-2: LoRA 어댑터 병합 (주석판)

### 어댑터 → 베이스 모델 병합 → 단일 모델 저장

> **입력**: `models/v1_law_adapter/` (Stage 3-1 결과)  
> **출력**: `models/v1_law/` (Gradio에서 바로 쓸 수 있는 단일 모델)

---

## 📌 왜 병합이 필요한가?

### 병합 전 (학습 직후 상태)
```
사용자 입력
   │
   ▼
[베이스 모델 로드 + 4bit 양자화]  ← VRAM 0.6GB, 느린 로드
   │
   + [LoRA 어댑터 부착]           ← PeftModel 래핑, 추론 속도↓
   │
   ▼
 응답
```

### 병합 후
```
사용자 입력
   │
   ▼
[단일 모델 로드]                 ← PeftModel 래핑 없음, 추론 빠름
   │
   ▼
 응답
```

**장점**: 
- Gradio 엔진 코드가 단순해짐 (`BaseEngine` 그대로 재사용 가능)
- 추론 속도 향상 (LoRA forward 오버헤드 제거)
- 배포/공유가 쉬움 (폴더 하나만 전달)

---

## ⚠️ 병합은 fp16으로

**4-bit 양자화 상태에서는 가중치 합산이 불가능**.  
$W_{merged} = W_{base} + \alpha \cdot A \cdot B$ 계산을 하려면 두 값이 **같은 수치 공간**에 있어야 하는데, 4-bit는 quantization constants와 scale factors로 얽혀 있어 바로 더할 수 없다.

따라서:
1. fp16으로 베이스 모델 로드 (일시적으로 VRAM 2~3GB 사용)
2. 어댑터 부착 → `merge_and_unload()`
3. 병합된 단일 모델 저장
4. 필요 시 나중에 4-bit로 다시 로드해서 사용


---
## 1️⃣ 환경 확인

**체크 포인트**:
- 어댑터 폴더 존재 여부 (이전 노트북 결과물)
- `adapter_config.json` 유무 (PEFT가 병합에 필요)
- GPU 사용 가능 여부

In [1]:
import sys, os

# 1-1. 프로젝트 루트를 sys.path에 추가
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel  # 어댑터 로드 및 병합용

from config.settings import BASE_MODEL_NAME, ADAPTER_DIR, MERGED_DIR

# 1-2. 어댑터 존재 여부를 assert로 미리 검증
#      병합 중간에 실패하면 VRAM만 낭비하므로 사전 검증이 중요
assert ADAPTER_DIR.exists(), f"❌ 어댑터 없음: {ADAPTER_DIR}"
assert (ADAPTER_DIR / "adapter_config.json").exists(), "❌ adapter_config.json 없음"

print(f"✅ 베이스:    {BASE_MODEL_NAME}")
print(f"✅ 어댑터:    {ADAPTER_DIR}")
print(f"✅ 저장 경로: {MERGED_DIR}")
print(f"✅ GPU:       {torch.cuda.get_device_name(0)}")

✅ 베이스:    LGAI-EXAONE/EXAONE-4.0-1.2B
✅ 어댑터:    c:\Users\user\Documents\fine_tunning\models\v1_law_adapter
✅ 저장 경로: c:\Users\user\Documents\fine_tunning\models\v1_law
✅ GPU:       NVIDIA GeForce RTX 3080


---
## 2️⃣ 베이스 모델 로드 (fp16)

**fp16을 선택한 이유**:
- 병합을 위해서는 양자화되지 않은 정밀도가 필요
- fp32는 VRAM 2배 소모 (1.2B × 4 bytes = 4.8GB)
- fp16이면 1.2B × 2 bytes = 2.4GB → 적절한 절충점
- `bf16`도 가능하지만 저장 후 다시 4-bit로 로드할 것이므로 fp16/bf16 선택이 최종 결과에 큰 영향 없음

> 💡 **gradient_checkpointing, device_map="auto"** 등은 학습용 옵션이므로 여기선 불필요.

In [2]:
# 2-1. 토크나이저 로드 (베이스에서 가져와도, 어댑터에서 가져와도 동일)
print("🔄 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2-2. 베이스 모델 로드
#   torch_dtype=float16 → 양자화 없이 fp16 정밀도 사용
#   device_map="auto"   → GPU 자동 배치
print("🔄 베이스 모델 로드 (fp16, 2~4분)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"✅ VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

🔄 토크나이저 로드...
🔄 베이스 모델 로드 (fp16, 2~4분)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

✅ VRAM: 2.56GB


---
## 3️⃣ 어댑터 부착

`PeftModel.from_pretrained(base, adapter_dir)` 는:
1. `adapter_config.json`을 읽어 LoRA 설정 복원
2. `adapter_model.safetensors`에서 학습된 A, B 행렬 로드
3. 원본 base_model의 target_modules 옆에 부착
4. PeftModel 래퍼로 감싸서 반환

이 시점의 모델은 **아직 병합되지 않은 상태** — LoRA 가중치가 별도로 붙어 있다.

In [3]:
# 3-1. 베이스 + 어댑터 = PeftModel
print(f"🔄 어댑터 로드: {ADAPTER_DIR}")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
print("✅ 부착 완료")

# 3-2. 학습 가능한 파라미터 확인
#      이 시점에서는 아직 LoRA 어댑터가 "얹혀진" 상태라
#      trainable: LoRA 파라미터만, total: 베이스+LoRA
model.print_trainable_parameters()

🔄 어댑터 로드: c:\Users\user\Documents\fine_tunning\models\v1_law_adapter
✅ 부착 완료
trainable params: 0 || all params: 1,294,628,608 || trainable%: 0.0000


---
## 4️⃣ 병합 (핵심!)

### `merge_and_unload()` 가 하는 일

각 target_module마다:
```
W_new = W_base + (alpha / r) * B @ A
```
을 실제로 계산해서 원래 Linear layer의 가중치를 갱신.
그리고 **LoRA 어댑터와 PeftModel 래퍼는 제거**.

결과: 순수 `LlamaForCausalLM`(또는 EXAONE) 객체 — 베이스 모델과 아키텍처가 완전히 동일하지만, 가중치만 파인튜닝된 버전.

In [4]:
# 4-1. 가중치 병합 및 어댑터 제거
print("🔄 가중치 병합 중...")
merged = model.merge_and_unload()

# 4-2. 타입 확인 — PeftModel이 아닌 원본 아키텍처가 나와야 정상
#      예: ExaoneForCausalLM, LlamaForCausalLM 등
print(f"✅ 병합 완료 - 타입: {type(merged).__name__}")

🔄 가중치 병합 중...
✅ 병합 완료 - 타입: Exaone4ForCausalLM


---
## 5️⃣ 단일 모델로 저장

### 저장 옵션 해설

| 옵션 | 의미 | 권장 이유 |
|------|------|-----------|
| `safe_serialization=True` | safetensors 포맷 사용 | pickle 취약점 없음, 로드 빠름 |
| `max_shard_size="2GB"` | 2GB마다 파일 분할 | HF Hub 업로드 제한 대응, 병렬 다운로드 |

저장되는 파일:
- `model-00001-of-00001.safetensors` — 모델 가중치
- `config.json` — 아키텍처 설정
- `generation_config.json` — 생성 기본값
- `tokenizer_config.json`, `tokenizer.json`, `special_tokens_map.json` — 토크나이저

In [5]:
# 5-1. 저장 디렉토리 생성
print(f"💾 저장 중: {MERGED_DIR}")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

# 5-2. 병합된 모델 저장
merged.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,   # safetensors 포맷 (권장)
    max_shard_size="2GB",       # 2GB마다 분할 저장
)

# 5-3. 토크나이저도 함께 저장 (모델과 같은 폴더에)
#   → 나중에 from_pretrained(MERGED_DIR) 한 방에 로드 가능
tokenizer.save_pretrained(str(MERGED_DIR))

print(f"\n✅ 저장 완료\n")

# 5-4. 저장된 파일 목록과 총 크기 확인
#      1.2B 모델이라면 약 2.4GB(fp16) 수준이 나와야 함
total_mb = 0
for f in sorted(os.listdir(MERGED_DIR)):
    size_mb = os.path.getsize(MERGED_DIR / f) / 1024 / 1024
    total_mb += size_mb
    print(f"   {f:45s} {size_mb:>8.1f} MB")
print(f"\n   📦 전체: {total_mb:.1f} MB")

💾 저장 중: c:\Users\user\Documents\fine_tunning\models\v1_law


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]


✅ 저장 완료

   chat_template.jinja                                0.0 MB
   config.json                                        0.0 MB
   generation_config.json                             0.0 MB
   model-00001-of-00002.safetensors                1896.2 MB
   model-00002-of-00002.safetensors                 544.1 MB
   model.safetensors.index.json                       0.0 MB
   tokenizer.json                                     7.5 MB
   tokenizer_config.json                              0.0 MB

   📦 전체: 2447.9 MB


---
## 6️⃣ 메모리 정리 + 재로드 테스트

### 왜 메모리 정리?
- fp16 베이스 모델 + PeftModel + 병합 모델이 모두 VRAM에 떠 있음 → **중복 점유**
- 재로드 테스트에서 4-bit로 다시 띄우려면 공간이 필요
- Python 참조를 `del`로 끊고 → `gc.collect()` → `cuda.empty_cache()` 순서가 표준

### 재로드 테스트의 의미
- 저장 → 로드 사이클이 **실제로 동작**하는지 검증
- PeftModel 래퍼 없이 한 줄로 로드되는지 확인 (Gradio 엔진에서 그대로 쓸 수 있다는 증거)

In [6]:
import gc

# 6-1. 거대 텐서 참조 해제
del base_model, model, merged

# 6-2. 가비지 컬렉터 실행
gc.collect()

# 6-3. CUDA 캐시 반환
torch.cuda.empty_cache()

print(f"✅ 정리 (VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB)")

✅ 정리 (VRAM: 0.01GB)


In [7]:
from transformers import BitsAndBytesConfig

# 7-1. 실제 배포 시 쓸 4-bit 양자화 설정
#      (BaseEngine과 동일한 설정 — 일관성 확인 목적)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 7-2. 병합 모델을 4-bit로 재로드
#   ⭐ PeftModel 없이 AutoModelForCausalLM 한 줄로 로드됨
#     → 병합이 잘 됐다는 증거, Gradio 엔진에서 그대로 사용 가능
test_model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR), quantization_config=bnb, device_map="auto",
)
test_tok = AutoTokenizer.from_pretrained(str(MERGED_DIR))
test_model.eval()

print(f"✅ 재로드 성공 (VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB)")

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

✅ 재로드 성공 (VRAM: 0.98GB)


### 실제 법률 질문으로 동작 확인

파인튜닝 효과를 확인하기 위해 학습 도메인(법률)의 질문을 던져본다.

**생성 파라미터 해설**:
| 파라미터 | 값 | 효과 |
|----------|-----|------|
| `temperature=0.1` | 낮음 | 답변이 일관되고 사실적 |
| `top_p=0.9` | 표준 | 상위 90% 확률 질량에서만 샘플링 |
| `do_sample=True` | 샘플링 사용 | 다양성 확보 |
| `repetition_penalty=1.1` | 약한 억제 | 같은 구절 반복 방지 |

In [8]:
from config.settings import SYSTEM_LEGAL

# 8-1. 테스트 질문
q = "임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?"

# 8-2. 프롬프트 구성 (ChatML)
messages = [
    {"role": "system", "content": SYSTEM_LEGAL},
    {"role": "user", "content": q},
]

# 8-3. 2단계 토크나이징 (BaseEngine 과 동일한 안정 패턴)
prompt = test_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tok(prompt, return_tensors="pt").to(test_model.device)

# 8-4. 추론 (학습 아님 → inference_mode 사용)
with torch.inference_mode():
    outputs = test_model.generate(
        inputs["input_ids"],
        max_new_tokens=300,           # 최대 출력 길이
        temperature=0.1,               # 낮게 → 일관된 답변
        top_p=0.9,                     # nucleus sampling
        do_sample=True,
        repetition_penalty=1.1,        # 반복 억제
        pad_token_id=test_tok.pad_token_id,
    )

# 8-5. 입력 프롬프트 부분을 잘라내고 생성 토큰만 디코드
gen = outputs[0][inputs["input_ids"].shape[1]:]
answer = test_tok.decode(gen, skip_special_tokens=True).strip()

print(f"[Q] {q}\n")
print(f"[A]\n{answer}")

[Q] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?

[A]
임차인이 보증금을 돌려받지 못하는 경우, 임차인은 임대인에게 보증금 반환을 청구할 수 있습니다. 만약 임대인이 보증금을 반환하지 않을 경우, 임차인은 민사소송을 통해 보증금 반환을 요구할 수 있으며, 이 과정에서 임대차 계약의 내용과 관련된 증거를 수집해야 합니다. 또한, 임차인은 임대인에게 보증금 반환을 요청하고, 필요한 경우 법원에 소송을 제기하여 법적 절차를 진행할 수 있습니다. 이러한 절차는 민법 제 628조 및 관련 판례에 따라 진행됩니다.


---
## 🎉 Stage 3 완료!

이제 파인튜닝 챗봇 실행 가능:
```bash
python -m ui.chat_finetuned
```

### 📝 검증 체크리스트

파인튜닝이 성공적이었는지 확인하는 방법:

1. **도메인 지식**: 법률 관련 질문에 전문용어(내용증명, 임차권등기명령, 소액심판 등)를 자연스럽게 사용하는가?
2. **톤 일관성**: SYSTEM_LEGAL에 설정한 페르소나대로 답변하는가?
3. **기본 능력 유지**: 도메인 외 질문("오늘 저녁 메뉴")에도 과도하게 법률적으로 답하지 않는가? (catastrophic forgetting 체크)
4. **비교**: `chat_basic.py`(순정)와 같은 질문을 넣어 차이가 느껴지는가?

In [9]:
import gc

# 9-1. 테스트 모델 정리
del test_model
gc.collect()
torch.cuda.empty_cache()

print("✅ 정리 완료")

✅ 정리 완료
